In [ ]:
# Install dependencies

# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# !pip install lightning
# !pip install xarray zarr gcsfs fsspec
# !pip install numpy pandas matplotlib scipy

This notebook uses a fully-connected neural network to predict global surface temperature from CO₂ and CH₄ concentration time series. It walks through data loading, normalization, model construction in PyTorch, training, and evaluation.

Originally developed by Weiwei Zhan; adapted to PyTorch by Gabriele Accarino.

# Import relevant Libraries

In [ ]:
# Core imports
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import xarray as xr
import numpy as np
import warnings
import gcsfs
import torch
import os

# custom utility library
from utils import *

fs = gcsfs.GCSFileSystem()

print(f"PyTorch      : {torch.__version__}")

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {DEVICE}")

warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 12, "figure.dpi": 100,
                     "legend.frameon": False, "axes.spines.top": False,
                     "axes.spines.right": False})


# 1. Load data and make some visualizations


In [ ]:
train_path = "gs://leap-persistent/jbusecke/data/climatebench/train_val/"
test_path  = "gs://leap-persistent/jbusecke/data/climatebench/test/"

Before training ML models, let's plot the ClimateBench dataset for quick checking.

ClimateBench is a spatial-temporal dataset that contains simulations generated by the NorESM2 model. It provides both historical simulations & future projections under different scenarios (e.g., ssp245).

Four future scenarios are plotted here: `ssp126, ssp245, ssp370, ssp585`.


1. ssp126 (Low): low population growth, high levels of education, and global cooperation to address environmental and social issues.

2. ssp245 (Medium): intermediate challenges to mitigation and adaptation - moderate population growth, intermediate levels of education, and a balanced emphasis on economic development and environmental sustainability.

3. ssp370 (High): continued high population growth, limited environmental regulations, and slow technological progress in achieving sustainability goals.

4. ssp585 (Very High): high population growth, limited technological innovation in sustainability, and high reliance on fossil fuels

In [ ]:
scenarios = ['historical','ssp126','ssp370','ssp585']
inputs = [os.path.join(train_path , f"inputs_{scenario}") for scenario in scenarios]
inputs.append(os.path.join(test_path, "inputs_ssp245"))
inputs.sort(key=lambda x:x.split('_')[-1])

outputs = [os.path.join(train_path , f"outputs_{scenario}") for scenario in scenarios]
outputs.append(os.path.join(test_path, "outputs_ssp245"))
outputs.sort(key=lambda x:x.split('_')[-1])

In [ ]:
inputs

In [ ]:
outputs

In [ ]:
# Data Loading
# Grid dimensions (matching NorESM2 / ClimateBench)
LAT   = np.linspace(-88.5, 88.5, 96)
LON   = np.linspace(0, 357.5, 144)
NLAT, NLON = len(LAT), len(LON)

def open_ds(path):
    store = fs.get_mapper(path)
    return xr.open_zarr(store, consolidated=True)

train_scenarios = ["historical", "ssp126", "ssp370", "ssp585"]
test_scenarios  = ["ssp245"]

In [ ]:
X_train_parts, y_train_parts = [], []
for f in train_scenarios:
    print(f'[Train] Processing scenario: {f}')
    X = open_ds(train_path + f"inputs_{f}.zarr")
    y = open_ds(train_path + f"outputs_{f}.zarr").mean("member")[["tas"]]
    X_train_parts.append(X) 
    y_train_parts.append(y)

X_train_xr = xr.concat(X_train_parts, dim="time")
X_train_xr = X_train_xr.assign_coords(time=X_train_xr.time)
y_train_xr = xr.concat(y_train_parts, dim="time")
y_train_xr = y_train_xr.assign_coords(time=y_train_xr.time)

X_test_parts, y_test_parts = [], []
for f in test_scenarios:
    print(f'[Test] Processing scenario: {f}')
    X_test = open_ds(test_path + f"inputs_{f}.zarr")
    y_test = open_ds(test_path + f"outputs_{f}.zarr").mean("member")[["tas"]]
    X_test_parts.append(X_test) 
    y_test_parts.append(y_test)

X_test_xr = xr.concat(X_test_parts, dim="time")
y_test_xr = xr.concat(y_test_parts, dim="time")

# train
co2_train = X_train_xr["CO2"].values          # (time, )           - input
ch4_train = X_train_xr["CH4"].values          # (time, )           - input 
tas_train = y_train_xr["tas"].values          # (time, lat, lon)   - target

# test
co2_test  = X_test_xr["CO2"].values           # (time, )           - input
ch4_test  = X_test_xr["CH4"].values           # (time, )           - input
tas_test  = y_test_xr["tas"].values           # (time, lat, lon)   - target

print(f"[Train] CO2 : {co2_train.shape},  CH4 : {ch4_train.shape},  tas : {tas_train.shape}")
print(f"[Test]  CO2 : {co2_test.shape},   CH4 : {ch4_test.shape},   tas : {tas_test.shape}")

## 1.1 Visualize input time series of CO2 and CH4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
colors  = ['tab:blue','tab:green','tab:purple','tab:orange','tab:red']

inputs = [os.path.join(train_path , f"inputs_{scenario}") for scenario in train_scenarios]
inputs.append(os.path.join(test_path, "inputs_ssp245"))
inputs.sort(key=lambda x:x.split('_')[-1])

for i,input in enumerate(inputs):

    label=input.split('_')[-1]#[:-3]
    X = open_dataset(input)
    x = X.time.data
    
    X['CO2'].plot(label=label, color=colors[i], linewidth=2, ax=axes[0])
    axes[0].set_ylabel("Cumulative anthropogenic CO2 \nemissions since 1850 (GtCO2)")
    
    X['CH4'].plot(label=label, color=colors[i], linewidth=2, ax=axes[1])
    axes[1].set_ylabel("Anthropogenic CH4 \nemissions (GtCH4 / year)")
    
axes[0].set_title('CO2')
axes[1].set_title('CH4')
axes[0].legend()
plt.tight_layout()

## 1.2 Visualize Surface Air Temperature Anomaly

In [ ]:
hist_years   = [1900, 1950, 2000]
future_years = [2020, 2050, 2100]

### Historical years

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, t in zip(axes, hist_years):
    im = ax.imshow(y_train_xr.sel(time=t).tas, origin="lower", cmap="RdBu_r", vmin=-3, vmax=3, extent=[0, 360, -90, 90], aspect="auto")
    ax.set_title(f"SSP245 tas — {t}"); ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    plt.colorbar(im, ax=ax, label="°C")
plt.suptitle("Surface Air Temperature anomaly (Training set)", fontweight="bold")
plt.tight_layout(); plt.show()

### Future years

In [ ]:
ssp_ids = {0: 'ssp126', 1: 'ssp370', 2: 'ssp585'}

scenario_idx = 0

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, t in zip(axes, future_years):
    im = ax.imshow(y_train_xr.sel(time=t).tas.values[1, ...], origin="lower", cmap="RdBu_r", vmin=-3, vmax=3, extent=[0, 360, -90, 90], aspect="auto")
    ax.set_title(f"{ssp_ids[scenario_idx]} tas — {t}"); ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    plt.colorbar(im, ax=ax, label="°C")
plt.suptitle("Surface Air Temperature anomaly (Training set)", fontweight="bold")
plt.tight_layout(); plt.show()

# 2. Dataset Class

In [ ]:
class ClimateDataset(Dataset):
    """
    Wraps raw NumPy arrays, handles normalization internally.

    Parameters
    ----------
    X        : (N, 2) array of [CO2, CH4] — raw, un-normalised
    y        : (N, lat*lon) array of tas — raw, un-normalised
    stats    : dict with keys 'X_mean','X_std','y_mean','y_std'.
               If None, stats are computed from this split (use for train only).
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, stats: dict | None = None):
        
        if stats is None:                      
            stats = {
                "X_mean": X.mean(axis=0),         # shape (2,) per-feature
                "X_std":  X.std(axis=0),
                "y_mean": y.mean(),               # scalar value (i.e., global mean)
                "y_std":  y.std(),
            }
            
        self.stats = stats                         

        X_norm = (X - stats["X_mean"]) / stats["X_std"]
        y_norm = (y - stats["y_mean"]) / stats["y_std"]

        self.X = torch.from_numpy(X_norm.astype(np.float32))
        self.y = torch.from_numpy(y_norm.astype(np.float32))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Stack the input variables and flatten the maps

In [ ]:
# Stack INPUT features (still raw — no normalisation here)
X_train_valid = np.column_stack([co2_train, ch4_train]).astype(np.float32)          # (train_years, 2)
X_test = np.column_stack([co2_test, ch4_test]).astype(np.float32)                   # (test_years, 2)

# Flatten spatial dimensions of the target to (time, 2). This 2 represents the stack of CO2 and CH4.
y_train_valid = tas_train.reshape(len(tas_train), NLAT * NLON).astype(np.float32)   # (train_years, 2)
y_test = tas_test.reshape(len(tas_test), NLAT * NLON).astype(np.float32)            # (test_years, 2)

## Validation data

Split X_train_valid to reserve 20% for validation

In [ ]:
SEED = 42
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=0.20, random_state=SEED)
X_train.shape, X_valid.shape, X_test.shape

## 2.2 Compute Stats

In [ ]:
stats = {
    "X_mean": X_train.mean(axis=0),         # shape (2,)  — per-feature
    "X_std":  X_train.std(axis=0),
    "y_mean": y_train.mean(),               # scalar — global
    "y_std":  y_train.std(),
}

## 2.3 Create Dataset Objects

In [ ]:
train_ds = ClimateDataset(X_train, y_train, stats=stats)
valid_ds = ClimateDataset(X_valid, y_valid, stats=stats)
test_ds  = ClimateDataset(X_test, y_test, stats=stats)

## 2.3 Create Dataloaders

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 3. Artificial Neural Network Setup

TODO — Build the MLP
Complete the __init__ and **forward** methods of the model class.

Architecture:

- Input size: 2 (CO₂ and CH₄ concentrations)
- Hidden layers: 2, each with 64 neurons
- Output size: NLAT * NLON
- Activation: ReLU after each hidden layer
- Dropout: 0.1 after each activation

What goes where:
- In __init__, declare every learnable layer (1 input, 2 hidded and 1 output nn.Linear modules), the activation function, and the dropout layer. 
- In forward, wire them together in sequence: input → hidden1 → ReLU → dropout → hidden2 → ReLU → dropout → output. 
- The output layer should have no activation, we want raw predictions for regression.


In [ ]:
class ClimateMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: complete

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        # TODO: complete
        return output

## 3.1 Create an instance of the model and inspect

In [ ]:
model = ClimateMLP().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

# 4. Training

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
LR         = 0.001
NUM_EPOCHS = 100
PATIENCE   = 10

# ── Loss & optimiser ──────────────────────────────────────────────────────────
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    batch_losses = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_b)
        loss  = loss_function(preds, y_b)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    # Validate
    model.eval()
    with torch.no_grad():
        val_batch_losses = [
            loss_function(model(X_b.to(DEVICE)), y_b.to(DEVICE)).item()
            for X_b, y_b in valid_loader
        ]
    val_loss = np.mean(val_batch_losses)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss   = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "/tmp/best_ann_pytorch.pt")
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | train-loss={train_loss:.4f} | val-loss={val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Reload best weights
model.load_state_dict(torch.load("/tmp/best_ann_pytorch.pt", map_location=DEVICE))
print(f"\nBest validation loss: {best_val_loss:.4f}")


## 4.1 Plot Training History

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label="Train loss", color="tab:blue", lw=2)
ax.plot(val_losses,   label="Val loss", color="tab:orange", lw=2, ls="--")
ax.set_xlabel("Epoch") 
ax.set_ylabel("MSE Loss")
ax.set_title("Training History") 
ax.legend()
plt.tight_layout() 
plt.show()

# 5. Evaluation

## 5.1 Make inference on the test set

In [ ]:
with torch.inference_mode():
    model.eval()
    test_preds = []
    for X_b, _ in test_loader:

        X_b  = X_b.to(DEVICE) # move to same device as model
        yhat = model(X_b)
        yhat_phys = yhat * train_ds.stats["y_std"] + train_ds.stats["y_mean"]
        preds_3d  = yhat_phys.reshape(-1, NLAT, NLON)   # reshape back to a map
        test_preds.append(preds_3d.cpu())               # keep as tensor 

    test_preds = torch.cat(test_preds, dim=0)           # (N_test, lat, lon)

test_preds.shape

## 5.2 Store predictions in an xarray Dataset

In [ ]:
yhat_test_xr = xr.Dataset(coords={'time': X_test_xr.time.values, 
                               'lat': X_test_xr.latitude.values, 
                               'lon': X_test_xr.longitude.values},
                       data_vars=dict(tas=(['time', 'lat', 'lon'], test_preds)))

## 5.3 Plot some predictions

In [ ]:
fig, axes = plt.subplots(figsize=(15,12), ncols=2, nrows=3)

yrs = [2030, 2050, 2100]
vmin, vmax    = -6, 6
cmap = 'RdBu_r'
y_test_xr.tas.sel(time=yrs[0]).plot(ax=axes[0,0], vmin=vmin, vmax=vmax,cmap=cmap)
yhat_test_xr.tas.sel(time=yrs[0]).plot(ax=axes[0,1], vmin=vmin, vmax=vmax,cmap=cmap)

y_test_xr.tas.sel(time=yrs[1]).plot(ax=axes[1,0], vmin=vmin, vmax=vmax,cmap=cmap)
yhat_test_xr.tas.sel(time=yrs[1]).plot(ax=axes[1,1], vmin=vmin, vmax=vmax,cmap=cmap)

y_test_xr.tas.sel(time=yrs[2]).plot(ax=axes[2,0], vmin=vmin, vmax=vmax,cmap=cmap)
yhat_test_xr.tas.sel(time=yrs[2]).plot(ax=axes[2,1], vmin=vmin, vmax=vmax,cmap=cmap)


for i, ax in enumerate(axes.flat):
    # left column: model prediction
    if i % 2 == 0:
        ax.set_title(f'tas model prediction (year = {yrs[i//2]})',fontweight='bold')
    # right column: truth tas from ssp245 simulations
    else:
        ax.set_title(f'tas ground-truth (year = {yrs[i//2]})',fontweight='bold')
plt.tight_layout()

## 5.4 Plot the Projected average temperature in NYC predicted by the model

In [ ]:
lat = 40.7128
lon = -74.0060%360

fig,ax = plt.subplots(figsize=(9,4))
y_test_xr.sel(lat=lat, lon=lon, method='nearest').tas.plot(marker='o', ax=ax, label='Ground-truth', linestyle='none')
yhat_test_xr.sel(lat=lat, lon=lon, method='nearest').tas.plot(ax=ax, label='prediction')

ax.set_title("Projected Surface Temperature at NYC (40.7°N, 74.0°W)")
ax.set_ylabel('temperature (°C)')
ax.legend()

plt.tight_layout()

## 5.5 Plot the Projected average global mean temperature

In [ ]:
fig,ax = plt.subplots(figsize=(9,4))
y_test_xr.tas.mean(dim=['lat', 'lon']).plot(marker='o', ax=ax, label='Ground-truth', linestyle='none')
yhat_test_xr.tas.mean(dim=['lat', 'lon']).plot(ax=ax, label='prediction')

ax.set_title("Global Mean Surface Temperature — SSP245")
ax.set_ylabel('temperature (°C)')
ax.legend()

plt.tight_layout()